# Feature engineering

Build the shared feature table from the supplied synthetic loan dataset. This notebook extracts features from borrower remarks, derives financial and credit variables, and exports the common training and test partitions for the group's models.

## 1. Load the raw dataset

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
from pathlib import Path

ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / 'data' / 'financial_loan_dataset_v2.csv').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from the project folder or its notebooks directory.')
DATA_DIR = ROOT / 'data'
TARGET = 'loan_default'
SEED = 42

In [3]:
df_raw = pd.read_csv(DATA_DIR / 'financial_loan_dataset_v2.csv')

print(f'Raw dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
print(f'Missing values: {df_raw.isna().sum().sum()}')
print(f"Unique remarks: {df_raw['customer_remark'].nunique():,}")
display(df_raw[TARGET].value_counts().rename(index={0: 'Non-default', 1: 'Default'}))

Raw dataset: 1,200 rows, 11 columns
Missing values: 0
Unique remarks: 1,188


loan_default
Non-default    952
Default        248
Name: count, dtype: int64

## 2. Extract features from borrower remarks

Qwen3.5 receives each unique remark as text. It scores sentiment and cash-flow strain in both thinking modes, and classifies loan purpose in non-thinking mode using seven general purpose categories and an Unknown option. The prompts use no default outcomes or simulator templates.

### 2.1 Local small language model extraction

[Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B) is an Apache 2.0 instruction-tuned language model. The notebook uses the text-only Q8_0 checkpoint from the [Ollama distribution](https://ollama.com/library/qwen3.5:0.8b). It runs locally through llama.cpp in both non-thinking and thinking modes. The first run verifies the checkpoint download. Later runs reuse cached scores for the same remarks and prompt settings. The model sees only the remark text.

The two prompts score fixed response labels for sentiment (positive, neutral, negative) and cash-flow strain (yes, no). We normalize the scores over the allowed label tokens. The features are positive minus negative sentiment score and the score for a yes response to the strain question. These scores summarize the wording and are not calibrated probabilities of default.

The thinking run lets the model generate up to 32 reasoning tokens per remark and prompt before scoring the same fixed labels. If the model reaches the cap, the notebook closes the thinking block and scores the answer. This measures a bounded local run. The exports contain the scores and labels.

For purpose, Qwen scores the numbered options. Unknown is available for remarks without a clear purpose. The label is exported for analysis. The logistic regression uses sentiment and strain scores as its two text predictors.

In [4]:
df = df_raw.copy()

import hashlib
import json
import os
import shutil
from contextlib import closing
from tempfile import NamedTemporaryFile
from urllib.request import urlopen

from llama_cpp import Llama, LogitsProcessorList, llama_supports_gpu_offload

MODEL_SHA256 = '889ea3b9143cdc32146c7d530ce5f3d452ec18069086f6f4167ce9ae72a9691f'
MODEL_URL = 'https://registry.ollama.ai/v2/library/qwen3.5/blobs/sha256:' + MODEL_SHA256
model_path = ROOT / '.cache' / 'qwen3.5-0.8b-q8_0.gguf'

SENTIMENT_PROMPT = (
    'Classify the financial sentiment as positive, neutral, or negative. Output one word. '
    'Examples: savings are comfortable => positive; the client asked about paperwork => neutral; '
    'the client is worried about paying bills => negative.'
)
STRAIN_PROMPT = (
    'Is the borrower under financial strain? Answer yes or no. '
    'Examples: "Savings comfortably cover recurring bills." -> no. '
    '"Asked about administrative steps." -> no. '
    '"Budget stretched by rising bills." -> yes. '
    '"Has no cash left and overdue bills." -> yes.'
)

PURPOSE_LABELS = (
    'Debt Consolidation', 'Business & Venture', 'Home Improvement', 'Education & Upskilling',
    'Medical & Emergency', 'Personal & Leisure', 'Vehicle Financing', 'Unknown',
)
PURPOSE_CHOICES = dict(zip('12345678', PURPOSE_LABELS, strict=True))
PURPOSE_PROMPT = (
    'Classify what the loan funds will be used for. Ignore the borrower\'s financial tone. '
    'Choose 8 if the loan purpose is unclear or missing. Output only the option number.\n'
    + '\n'.join(f'{choice}. {label}' for choice, label in PURPOSE_CHOICES.items())
)

THINKING_BUDGET = 32


def score_remark_labels(model, remarks, instruction, labels, thinking_budget=0):
    label_tokens = []
    for label in labels:
        forms = (label, ' ' + label)
        token_ids = {
            tokens[0] for form in forms
            if len(tokens := model.tokenize(form.encode(), add_bos=False)) == 1
        }
        if not token_ids:
            raise ValueError(f'Expected a single-token spelling of {label!r}.')
        label_tokens.append(sorted(token_ids))


    reason_close = model.tokenize(b'</think>', add_bos=False, special=True)
    separator = model.tokenize(b'\n\n', add_bos=False)
    end_tokens = {model.token_eos(), *model.tokenize(b'<|im_end|>', add_bos=False, special=True)}
    if len(reason_close) != 1:
        raise ValueError('Expected a single Qwen thinking end token.')

    rows = []
    capped = 0
    for idx, remark in enumerate(remarks, start=1):
        prompt = (
            f'<|im_start|>system\n{instruction}<|im_end|>\n'
            f'<|im_start|>user\nNote: {remark}<|im_end|>\n<|im_start|>assistant\n<think>\n'
        )
        if thinking_budget == 0:
            prompt += '\n</think>\n\n'
        tokens = model.tokenize(prompt.encode(), add_bos=False, special=True)
        capture_answer = thinking_budget == 0

        def capture_scores(_, logits):
            if capture_answer:
                label_logits = np.array([np.max(logits[ids]) for ids in label_tokens])
                weights = np.exp(label_logits - label_logits.max())
                rows.append(weights / weights.sum())
            return logits

        processor = LogitsProcessorList([capture_scores])
        with closing(model.generate(tokens, temp=0, logits_processor=processor)) as generated:
            if thinking_budget:
                for _ in range(thinking_budget):
                    token = next(generated)
                    if token == reason_close[0]:
                        closing_tokens = [token] + separator
                        break
                    if token in end_tokens:
                        closing_tokens = reason_close + separator
                        break
                else:
                    capped += 1
                    closing_tokens = [token] + reason_close + separator
            else:
                next(generated)
        if thinking_budget:
            # The generator yielded the final token without evaluating it. Finish the thinking
            # block on the existing model state and read only the first answer logits.
            capture_answer = True
            model.eval(closing_tokens)
            model.sample(temp=0)
            if idx % 400 == 0:
                print(f'Thinking scores complete for {idx:,} of {len(remarks):,} remarks.', flush=True)

    return pd.DataFrame(rows, index=remarks, columns=labels), capped


def cached_qwen_scores(remarks, prompts, settings, score_notes):
    cache_inputs = [MODEL_SHA256, *prompts, settings, remarks.tolist()]
    cache_key = hashlib.sha256(json.dumps(cache_inputs, ensure_ascii=False).encode('utf-8')).hexdigest()[:20]
    score_path = model_path.parent / f'qwen-remark-scores-{cache_key}.csv'
    if score_path.is_file():
        scores = pd.read_csv(score_path, index_col='customer_remark', keep_default_na=False, float_precision='round_trip')
        if not scores.index.equals(remarks):
            raise ValueError('Cached Qwen remarks do not match. Delete the score cache and rerun.')
        print(f'Loaded Qwen scores ({settings}) for {len(scores):,} unique remarks.')
        return scores

    if not model_path.exists():
        model_path.parent.mkdir(exist_ok=True)
        download = model_path.with_suffix('.part')
        print('Downloading the 812 MB Qwen3.5 checkpoint.')
        with urlopen(MODEL_URL, timeout=60) as response, download.open('wb') as file:
            shutil.copyfileobj(response, file)
        with download.open('rb') as file:
            if hashlib.file_digest(file, 'sha256').hexdigest() != MODEL_SHA256:
                raise ValueError('The downloaded Qwen3.5 checkpoint failed its SHA-256 check.')
        download.replace(model_path)

    with closing(Llama(
        model_path=str(model_path), n_ctx=256, n_batch=256,
        n_threads=min(os.cpu_count() or 1, 4), n_threads_batch=min(os.cpu_count() or 1, 4),
        n_gpu_layers=-1 if llama_supports_gpu_offload() else 0, verbose=False,
    )) as slm:
        scores = score_notes(slm)

    with NamedTemporaryFile(mode='w', encoding='utf-8', newline='', dir=score_path.parent, delete=False) as cache_file:
        scores.to_csv(cache_file)
    Path(cache_file.name).replace(score_path)
    return scores


def qwen_sentiment_and_strain(remarks, thinking_budget=0):
    mode = f'thinking-{thinking_budget}' if thinking_budget else 'non-thinking'

    def score_notes(slm):
        sentiment, sentiment_capped = score_remark_labels(
            slm, remarks, SENTIMENT_PROMPT, ('positive', 'neutral', 'negative'), thinking_budget
        )
        print(f'Scored {mode} sentiment for {len(sentiment):,} unique remarks.', flush=True)
        strain, strain_capped = score_remark_labels(
            slm, remarks, STRAIN_PROMPT, ('yes', 'no'), thinking_budget
        )
        print(f'Scored {mode} cash-flow strain for {len(strain):,} unique remarks.', flush=True)
        if thinking_budget:
            print(f'Reached the {thinking_budget}-token cap for {sentiment_capped:,} sentiment prompts '
                  f'and {strain_capped:,} strain prompts.')
        return pd.DataFrame({
            'slm_sentiment_label': sentiment.idxmax(axis=1),
            'slm_sentiment_score': sentiment['positive'] - sentiment['negative'],
            'slm_risk_category': strain.idxmax(axis=1).map({'yes': 'Strain indicated', 'no': 'Strain unclear'}),
            'slm_strain_score': strain['yes'],
        })

    return cached_qwen_scores(
        remarks, (SENTIMENT_PROMPT, STRAIN_PROMPT), f'{mode}-label-scores-v1', score_notes
    )

def score_purpose(slm):
    scores, _ = score_remark_labels(slm, unique_remarks, PURPOSE_PROMPT, tuple(PURPOSE_CHOICES))
    print(f'Scored loan purpose for {len(scores):,} unique remarks.')
    return scores.idxmax(axis=1).map(PURPOSE_CHOICES).to_frame('customer_segment')


unique_remarks = pd.Index(df['customer_remark'].fillna('').unique(), name='customer_remark')
slm_features = qwen_sentiment_and_strain(unique_remarks)
thinking_features = qwen_sentiment_and_strain(unique_remarks, thinking_budget=THINKING_BUDGET)
thinking_features = thinking_features.rename(columns=lambda name: name.replace('slm_', 'slm_thinking_', 1))
purpose_features = cached_qwen_scores(unique_remarks, (PURPOSE_PROMPT,), 'purpose-label-scores-v1', score_purpose)
remark_features = pd.concat([slm_features, thinking_features, purpose_features], axis=1)
df = df.join(remark_features.reindex(df['customer_remark'].fillna('')).set_axis(df.index))


Loaded Qwen scores (non-thinking-label-scores-v1) for 1,188 unique remarks.


Loaded Qwen scores (thinking-32-label-scores-v1) for 1,188 unique remarks.


ggml_vulkan: Found 2 Vulkan devices:


ggml_vulkan: 0 = NVIDIA GeForce RTX 4050 Laptop GPU (NVIDIA) | uma: 0 | fp16: 1 | bf16: 1 | fp4: 0 | warp size: 32 | shared memory: 49152 | int dot: 1 | matrix cores: NV_coopmat2


ggml_vulkan: 1 = Intel(R) RaptorLake-S Mobile Graphics Controller (Intel Corporation) | uma: 1 | fp16: 1 | bf16: 0 | fp4: 0 | warp size: 32 | shared memory: 32768 | int dot: 1 | matrix cores: none


Scored loan purpose for 1,188 unique remarks.


## 3. Derive tabular features

Estimate repayment burden using a 36-month principal-only installment proxy. Add delinquency indicators, credit interactions, log transforms, credit-score tiers, and an ordinal employment feature. The employment encoding uses Contract as the reference category.

The 36-month assumption makes total_burden_ratio exactly equal to debt_to_income_ratio + loan_to_income_ratio / 3. Keep all three available for group models and omit at least one of them from a linear model whose coefficients you plan to interpret. The v2 simulator already incorporates loan-to-income into its DTI input, so treat total_burden_ratio as a synthetic proxy.

In [5]:
# (A) Financial Capacity & Debt Service Ratios (Crucial for Tree Models, XGBoost, and Linear Models)
df['loan_to_income_ratio'] = df['loan_amount'] / df['annual_income']
df['monthly_income'] = df['annual_income'] / 12.0
df['estimated_monthly_debt'] = df['monthly_income'] * df['debt_to_income_ratio']
df['estimated_monthly_installment'] = df['loan_amount'] / 36.0  # 36-month principal-only installment proxy
df['installment_to_income_ratio'] = df['estimated_monthly_installment'] / df['monthly_income']
df['total_burden_ratio'] = df['debt_to_income_ratio'] + df['installment_to_income_ratio']
df['residual_monthly_income'] = df['monthly_income'] * (1.0 - df['debt_to_income_ratio']) - df['estimated_monthly_installment']

# (B) Behavioral Delinquency & Non-Linear Interaction Terms (Crucial for XGBoost, Random Forest, MLP)
df['late_payment_intensity'] = df['late_payments_12m'] / (df['account_tenure_months'] / 12.0 + 1.0)
df['has_late_payments'] = (df['late_payments_12m'] > 0).astype(int)
df['chronic_delinquency_flag'] = (df['late_payments_12m'] >= 2).astype(int)
df['credit_utilization_stress'] = df['debt_to_income_ratio'] / (df['credit_score'] / 850.0)
df['risk_adjusted_leverage'] = (df['loan_to_income_ratio'] * df['debt_to_income_ratio']) / (df['credit_score'] / 850.0)

# (C) Log Transforms (income and loan amount are right-skewed; tenure is evaluated separately)
df['log_annual_income'] = np.log1p(df['annual_income'])
df['log_loan_amount'] = np.log1p(df['loan_amount'])
df['log_tenure_months'] = np.log1p(df['account_tenure_months'])

# (D) Domain Discretization & Ordinal Encoding (Crucial for Decision Trees, Random Forest, and k-NN)
# FICO Credit Score Tiers: 0=Poor (<580), 1=Fair (580-669), 2=Good (670-739), 3=Very Good (740-799), 4=Exceptional (800+)
df['credit_score_tier'] = pd.cut(
    df['credit_score'],
    bins=[0, 579, 669, 739, 799, 900],
    labels=[0, 1, 2, 3, 4]
).astype(int)

emp_stability_map = {'Full-Time': 3, 'Self-Employed': 2, 'Contract': 1, 'Part-Time': 0}
df['employment_stability_score'] = df['employment_status'].map(emp_stability_map)

In [6]:
employment_dummies = pd.get_dummies(
    df['employment_status'].astype(pd.CategoricalDtype(
        categories=['Contract', 'Full-Time', 'Part-Time', 'Self-Employed'],
        ordered=True,
    )),
    prefix='emp',
    drop_first=True,
    dtype=int,
)
df = df.join(employment_dummies)

## 4. Create the shared split and inspect the training features

The supplied stratified split uses seed 42 and assigns about 20% of each target class to the test set. All models should use these same partitions. Sentiment correlations and feature diagnostics below use the training partition.

Fit any model-specific imputation or scaling on training data inside cross-validation. Use loan_default as the target and leave customer_id and customer_remark out of the predictor matrix.

In [7]:
# Stratified 80/20 Train-Test Split (seed=42)
def stratified_train_test_split(data, target_col='loan_default', test_size=0.20, seed=42):
    rng = np.random.RandomState(seed)
    train_indices, test_indices = [], []
    for cls_val in sorted(data[target_col].unique()):
        cls_idx = np.flatnonzero(data[target_col].to_numpy() == cls_val)
        rng.shuffle(cls_idx)
        n_test = int(round(len(cls_idx) * test_size))
        test_indices.extend(cls_idx[:n_test])
        train_indices.extend(cls_idx[n_test:])
    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    return data.iloc[train_indices].reset_index(drop=True), data.iloc[test_indices].reset_index(drop=True)

train_df, test_df = stratified_train_test_split(df, TARGET, test_size=0.20, seed=SEED)

In [8]:
split_summary = pd.DataFrame(
    {
        'Rows': [len(train_df), len(test_df)],
        'Defaults': [train_df[TARGET].sum(), test_df[TARGET].sum()],
        'Default rate': [train_df[TARGET].mean(), test_df[TARGET].mean()],
    },
    index=['Train', 'Test'],
)
display(split_summary.round(4))

for feature in ['slm_risk_category', 'slm_thinking_risk_category', 'customer_segment']:
    summary = train_df.groupby(feature)[TARGET].agg(Rows='size', Default_rate='mean')
    display(summary.sort_values('Default_rate', ascending=False).round(4))

note_scores = [
    'slm_sentiment_score', 'slm_strain_score',
    'slm_thinking_sentiment_score', 'slm_thinking_strain_score',
]
display(train_df[note_scores].corrwith(train_df[TARGET]).round(4).rename('Training correlation'))

log_columns = {
    'annual_income': 'log_annual_income',
    'loan_amount': 'log_loan_amount',
    'account_tenure_months': 'log_tenure_months',
}
skewness = pd.DataFrame(
    {
        'Raw': [train_df[column].skew() for column in log_columns],
        'Log transformed': [train_df[column].skew() for column in log_columns.values()],
    },
    index=list(log_columns),
)
display(skewness.round(4))

,Rows,Defaults,Default rate
Train,960,198,0.2062
Test,240,50,0.2083


,Rows,Default_rate
slm_risk_category,,
Strain indicated,34,0.4412
Strain unclear,926,0.1976


,Rows,Default_rate
slm_thinking_risk_category,,
Strain unclear,960,0.2062


,Rows,Default_rate
customer_segment,,
Debt Consolidation,23,0.3478
Medical & Emergency,163,0.2331
Unknown,314,0.2229
Home Improvement,36,0.2222
Business & Venture,175,0.2057
Vehicle Financing,80,0.2000
Education & Upskilling,84,0.1429
Personal & Leisure,85,0.1176


slm_sentiment_score            -0.2678
slm_strain_score                0.2348
slm_thinking_sentiment_score   -0.2714
slm_thinking_strain_score       0.0210
Name: Training correlation, dtype: float64

,Raw,Log transformed
annual_income,1.4382,0.0433
loan_amount,1.7412,0.0671
account_tenure_months,0.0749,-1.0391


## 5. Export data for the group

loan_features.csv contains the complete feature table. train.csv and test.csv have the same columns and hold the shared stratified split.

In [9]:
for filename, table in (
    ('loan_features.csv', df), ('train.csv', train_df), ('test.csv', test_df),
):
    table.to_csv(DATA_DIR / filename, index=False)
    print(f'Wrote {filename}: {table.shape[0]:,} rows, {table.shape[1]} columns')
print(f"Unique remarks scored by Qwen3.5: {len(unique_remarks):,}")

Wrote loan_features.csv: 1,200 rows, 40 columns
Wrote train.csv: 960 rows, 40 columns
Wrote test.csv: 240 rows, 40 columns
Unique remarks scored by Qwen3.5: 1,188


### Candidate feature sets for the other models

Tree models can use financial ratios, delinquency flags, credit tiers, and risk interactions directly. Scale the MLP and k-NN inputs using training data, and use the smaller feature set for k-NN. The logistic regression feature sets and training-set VIF checks are in [04b_model_logistic_regression.ipynb](04b_model_logistic_regression.ipynb).

In [10]:
FEATURE_SETS_BY_MODEL = {'decision_tree_and_random_forest': ['age',
                                     'annual_income',
                                     'credit_score',
                                     'credit_score_tier',
                                     'loan_amount',
                                     'debt_to_income_ratio',
                                     'late_payments_12m',
                                     'account_tenure_months',
                                     'loan_to_income_ratio',
                                     'total_burden_ratio',
                                     'residual_monthly_income',
                                     'late_payment_intensity',
                                     'has_late_payments',
                                     'chronic_delinquency_flag',
                                     'credit_utilization_stress',
                                     'employment_stability_score',
                                     'slm_sentiment_score',
                                     'slm_strain_score'],
 'gradient_boosting': ['age',
                       'annual_income',
                       'credit_score',
                       'credit_score_tier',
                       'loan_amount',
                       'debt_to_income_ratio',
                       'late_payments_12m',
                       'account_tenure_months',
                       'loan_to_income_ratio',
                       'total_burden_ratio',
                       'residual_monthly_income',
                       'late_payment_intensity',
                       'credit_utilization_stress',
                       'risk_adjusted_leverage',
                       'employment_stability_score',
                       'slm_sentiment_score',
                       'slm_strain_score'],
 'mlp': ['age',
         'log_annual_income',
         'credit_score',
         'log_loan_amount',
         'debt_to_income_ratio',
         'late_payments_12m',
         'log_tenure_months',
         'loan_to_income_ratio',
         'total_burden_ratio',
         'late_payment_intensity',
         'credit_utilization_stress',
         'emp_Full-Time',
         'emp_Part-Time',
         'emp_Self-Employed',
         'slm_sentiment_score',
         'slm_strain_score'],
 'knn': ['age',
         'log_annual_income',
         'credit_score',
         'loan_to_income_ratio',
         'debt_to_income_ratio',
         'late_payments_12m',
         'log_tenure_months',
         'employment_stability_score',
         'slm_sentiment_score',
         'slm_strain_score']}